# NB-InvSym 2: Champion-pool combination search (regression)

Takes the **best inversion-symmetry descriptors** (from nb_invsym_01) and pools them with the other
proven feature groups, then does an **exhaustive** combination search at sizes 4-6 (every C(pool,k)
subset, not greedy) and switches to **greedy** forward selection beyond 6. The point: greedy can miss
a strong 5-feature combo that needs two features added together; exhaustive at 4-6 catches those, and
beyond 6 the combinatorics are too large so greedy takes over.

**Pool = three groups, capped at ~15 total so C(15,6)=5005 stays tractable:**
- best ~10 inversion-symmetry features (ranked by alpha-correlation, built here same as nb_invsym_01)
- baseline orbital: E_pfrac_VBM, E_pfrac_CBM
- top GRDF/env + elemental features (built with the nb_expand functions, e.g. zw_grdf_*, radius_mean)

**Targets:** on the **99** slice try `alpha_max` AND `alpha_mean` (skip min here, per request); on the
**124** slice `alpha_max` only. Exhaustive search uses fast 5-fold; finalists confirmed with LOO.

**Heavy by design** - exhaustive 4-6 over a 15-pool is ~9,400 subsets x (5-fold = 5 fits). Accepted.
Place in `Keshav-DDP/Broken-inversion-symmetry/`.

## Cell 1: Config + imports

In [ ]:
import os, glob, warnings, itertools
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from scipy.special import spherical_jn
from scipy.stats import spearmanr

BASE_DIR = os.path.abspath("..")
SET_DIRS = {
    "rashba":      os.path.join(BASE_DIR, "Inverse-design", "rashba"),
    "dresselhaus": os.path.join(BASE_DIR, "Inverse-design", "dresselhaus"),
    "unmatched":   os.path.join(BASE_DIR, "Inverse-design", "unmatched"),
}
DESC_DIR = os.path.join(BASE_DIR, "Weight-contribution", "contribution-model")
DESC_CSVS = {
    "rashba":      os.path.join(DESC_DIR, "desc_E_p1.0_rashba.csv"),
    "dresselhaus": os.path.join(DESC_DIR, "desc_E_p1.0_dresselhaus.csv"),
    "unmatched":   os.path.join(DESC_DIR, "desc_E_p1.0_unmatched.csv"),
}
RESULTS_DIR = os.path.join(".", "invsym-results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("config ready")

## Cell 2: Master table + structures (with all three targets)

alpha = max per uid (headline), plus alpha_mean and alpha_min for the multi-target runs.

In [ ]:
parts = []
for s, csv in DESC_CSVS.items():
    c = pd.read_csv(csv)
    need = ["uid", "Formula", "alpha_R", "E_pfrac_VBM", "E_pfrac_CBM"]
    c = c[need].copy(); c["set"] = s
    if s == "unmatched":
        c["alpha_R"] = 0.0
    parts.append(c)
raw = pd.concat(parts, ignore_index=True)
g = raw.groupby("uid")
master = pd.DataFrame({
    "Formula":     g["Formula"].first(),
    "alpha":       g["alpha_R"].max(),
    "alpha_mean":  g["alpha_R"].mean(),
    "alpha_min":   g["alpha_R"].min(),
    "E_pfrac_VBM": g["E_pfrac_VBM"].first(),
    "E_pfrac_CBM": g["E_pfrac_CBM"].first(),
    "set":         g["set"].apply(lambda v: "rashba" if "rashba" in set(v)
                                  else ("dresselhaus" if "dresselhaus" in set(v) else "unmatched")),
    "is_split":    g["set"].apply(lambda v: int(any(x in ("rashba", "dresselhaus") for x in v))),
}).reset_index()

def find_folder(uid, vdir):
    for folder in glob.glob(os.path.join(vdir, "*")):
        name = os.path.basename(folder); idx = name.rfind("-")
        if idx != -1 and name[idx+1:] == uid:
            return folder
    return None

def load_std(folder):
    p = os.path.join(folder, "POSCAR_std")
    return Structure.from_file(p) if os.path.exists(p) else None

structures = {}
for _, row in master.iterrows():
    folder = find_folder(row["uid"], SET_DIRS[row["set"]])
    s = load_std(folder) if folder else None
    if s is not None and max(site.specie.Z for site in s) > 3:
        structures[row["uid"]] = s
master = master[master["uid"].isin(structures)].reset_index(drop=True)
print("compounds:", len(master), master["set"].value_counts().to_dict())

## Cell 3: GRDF + elemental features (the nb_expand functions, verbatim, so names match the project)

Rebuilt here rather than read from a CSV so they are guaranteed consistent with our uids. Same RC=6.0,
Gaussian centres, Z-weighting, AFS, simple geometry, and elemental properties as nb0/nb_expand.

In [ ]:
RC = 6.0
GAUSS_CENTERS = [2.0, 2.6, 3.2, 3.8, 4.4, 5.0]; GAUSS_SIGMA = 0.5
COS_K = [np.pi/RC*m for m in (1, 2, 3)]; BESSEL_K = np.pi/RC; BESSEL_N = [0, 1, 2]
AFS_MU, AFS_SIGMA = 2.8, 0.8; AFS_HARMONICS = [1, 2, 3]; AFS_RANG = 4.0; FIRST_SHELL = 3.5
def fc(r, rc=RC): return np.where(r <= rc, 0.5*(np.cos(np.pi*r/rc)+1.0), 0.0)
def b_gauss(r, mu, sig=GAUSS_SIGMA): return np.exp(-((r-mu)**2)/(2*sig**2))
def b_cos(r, k): return np.cos(k*r)
def b_bessel(r, n, k=BESSEL_K): return spherical_jn(n, k*r)

def neighbor_data(s, rc=RC):
    alln = s.get_all_neighbors(rc); out = []
    for i, site in enumerate(s):
        ci = site.coords; zi = site.specie.Z; d=[]; v=[]; zj=[]
        for nb in alln[i]:
            if nb.nn_distance <= 1e-3: continue
            d.append(nb.nn_distance); v.append(np.array(nb.coords)-ci); zj.append(nb.specie.Z)
        out.append(dict(zi=zi, d=np.array(d),
                        v=np.array(v) if v else np.zeros((0,3)), zj=np.array(zj)))
    return out

def grdf(nd, basis, weighted=False):
    per = []
    for a in nd:
        d = a["d"]
        if d.size == 0: per.append(0.0); continue
        w = basis(d)*fc(d)
        if weighted: w = w*np.sqrt(a["zi"]*a["zj"])
        per.append(float(w.sum()))
    per = np.array(per); return per.mean(), per.std()

def afs(nd):
    res = {m: [] for m in AFS_HARMONICS}
    for a in nd:
        d, v = a["d"], a["v"]; mask = d <= AFS_RANG; d, v = d[mask], v[mask]
        if d.size < 2:
            for m in AFS_HARMONICS: res[m].append(0.0)
            continue
        wr = np.exp(-((d-AFS_MU)**2)/(2*AFS_SIGMA**2))*fc(d)
        acc = {m: 0.0 for m in AFS_HARMONICS}
        for j in range(len(d)):
            for k in range(j+1, len(d)):
                c = np.clip(np.dot(v[j], v[k])/(d[j]*d[k]), -1, 1); th = np.arccos(c); ww = wr[j]*wr[k]
                for m in AFS_HARMONICS: acc[m] += ww*np.cos(m*th)
        for m in AFS_HARMONICS: res[m].append(acc[m])
    return {m: (np.array(res[m]).mean(), np.array(res[m]).std()) for m in AFS_HARMONICS}

def simple_geom(nd):
    coord=[]; bonds=[]; angles=[]
    for a in nd:
        d, v = a["d"], a["v"]; m = d <= FIRST_SHELL; coord.append(int(m.sum()))
        dd, vv = d[m], v[m]; bonds.extend(dd.tolist())
        for j in range(len(dd)):
            for k in range(j+1, len(dd)):
                c = np.clip(np.dot(vv[j], vv[k])/(dd[j]*dd[k]), -1, 1); angles.append(np.degrees(np.arccos(c)))
    coord=np.array(coord); bonds=np.array(bonds); angles=np.array(angles)
    return dict(coord_num=coord.mean(),
                bond_mean=bonds.mean() if bonds.size else 0.0, bond_std=bonds.std() if bonds.size else 0.0,
                angle_mean=angles.mean() if angles.size else 0.0, angle_std=angles.std() if angles.size else 0.0)

def elemental_props(s):
    els = list(set(s.species)); zs=[e.Z for e in els]; ms=[float(e.atomic_mass) for e in els]
    rad=[float(e.atomic_radius) if e.atomic_radius else 1.0 for e in els]
    return dict(max_Z=max(zs), max_Z4=max(z**4 for z in zs), max_mass=max(ms),
                radius_mean=np.mean(rad), radius_diff=(max(rad)-min(rad)) if len(rad)>1 else 0.0,
                n_elements=len(els), mean_Z=np.mean(zs), sum_Z4=sum(z**4 for z in zs))
print("grdf/elemental fns ready")

## Cell 4: Inversion-symmetry features (the builders from nb_invsym_01, verbatim)

Same adaptive Rc, ANY-species pairing, matched-fraction + distance/angle residuals, plain/Z^4,
mean/max, inversion + mirror + global-centre + structural. Kept identical so the pooled run is
consistent with nb_invsym_01.

In [ ]:
RC_TARGET = 3.5; RC_SAFETY = 0.95; TOLS = [0.2, 0.3, 0.5, 0.75]

def wrap_radius(s):
    abc = np.array(s.lattice.abc); vac = int(np.argmax(abc))
    inplane = [ax for ax in range(3) if ax != vac]; M = s.lattice.matrix
    a1, a2 = M[inplane[0]], M[inplane[1]]
    min_repeat = min(np.linalg.norm(a1), np.linalg.norm(a2),
                     np.linalg.norm(a1 + a2), np.linalg.norm(a1 - a2))
    return 0.5 * min_repeat, vac

def adaptive_rc(s):
    wr, vac = wrap_radius(s); return min(RC_TARGET, RC_SAFETY * 2 * wr), vac

def inv_neighbors(s, rc):
    alln = s.get_all_neighbors(rc); out = []
    for i, site in enumerate(s):
        ci = site.coords; v, zj = [], []
        for nb in alln[i]:
            if nb.nn_distance <= 1e-3: continue
            v.append(np.array(nb.coords) - ci); zj.append(nb.specie.Z)
        out.append(dict(zi=site.specie.Z, v=np.array(v) if v else np.zeros((0,3)), zj=np.array(zj)))
    return out

def _agg(vals, prefix, feat):
    v = np.array(vals, dtype=float)
    feat[f"{prefix}_mean"] = float(v.mean()) if v.size else 0.0
    feat[f"{prefix}_max"]  = float(v.max())  if v.size else 0.0

def _op(nd, tol, flip):
    frac, frac_z4, dres, dres_z4, ares, ares_z4 = [], [], [], [], [], []
    for a in nd:
        V = a["v"]
        if len(V) < 2: continue
        norms = np.linalg.norm(V, axis=1); Zi = a["zi"]; Zj = a["zj"]
        m, d, ang, w4 = [], [], [], []
        for j in range(len(V)):
            target = V[j]*flip if flip is not None else -V[j]
            dd = np.linalg.norm(V - target, axis=1); b = int(np.argmin(dd))
            m.append(1.0 if dd[b] <= tol else 0.0); d.append(abs(norms[b]-norms[j]))
            ct = np.clip(np.dot(V[b], target)/(norms[b]*norms[j]+1e-12), -1, 1); ang.append(np.arccos(ct))
            w4.append(max(Zi, Zj[j])**4)
        m=np.array(m); d=np.array(d); ang=np.array(ang); w4=np.array(w4)
        frac.append(1.0-m.mean()); frac_z4.append(1.0-(m*w4).sum()/w4.sum())
        dres.append(d.mean()); dres_z4.append((d*w4).sum()/w4.sum())
        ares.append(ang.mean()); ares_z4.append((ang*w4).sum()/w4.sum())
    return dict(frac=frac, frac_z4=frac_z4, dres=dres, dres_z4=dres_z4, ares=ares, ares_z4=ares_z4)

def build_invsym(s):
    rc, vac = adaptive_rc(s); nd = inv_neighbors(s, rc); feat = {}
    flip = np.ones(3); flip[vac] = -1.0
    for opname, fl in [("inv", None), ("mir", flip)]:
        for tol in TOLS:
            r = _op(nd, tol, fl); t = f"{tol:.2f}".replace(".", "")
            _agg(r["frac"],    f"{opname}_broken_t{t}",    feat)
            _agg(r["frac_z4"], f"{opname}_broken_z4_t{t}", feat)
        r0 = _op(nd, TOLS[0], fl)
        for nm in ["dres", "dres_z4", "ares", "ares_z4"]:
            _agg(r0[nm], f"{opname}_{nm}", feat)
    # global best-fit inversion centre (3x3)
    cart = s.cart_coords; Z = np.array([a.specie.Z for a in s], dtype=float); L = s.lattice
    best = np.inf; best_z4 = np.inf
    for fa in np.linspace(0, 0.5, 3):
        for fb in np.linspace(0, 0.5, 3):
            c = L.get_cartesian_coords([fa, fb, 0.5])
            res = np.array([np.linalg.norm(cart-(2*c-cart[i]), axis=1).min() for i in range(len(s))])
            best = min(best, res.mean()); best_z4 = min(best_z4, (res*Z**4).sum()/(Z**4).sum())
    feat["glob_inv_resid"] = float(best); feat["glob_inv_resid_z4"] = float(best_z4)
    # structural
    zc = cart[:, vac]; syms = np.array([a.specie.symbol for a in s]); Zs = np.array([a.specie.Z for a in s])
    heavy = syms==syms[np.argmax(Zs)]; light = syms==syms[np.argmin(Zs)]
    feat["buckling"] = float(abs(zc[heavy].mean()-zc[light].mean())) if (heavy.any() and light.any()) else 0.0
    asym = [abs(a["v"][:, vac].sum()) for a in nd if len(a["v"])]
    _agg(asym, "topbot_asym", feat)
    feat["wrap_radius"] = float(wrap_radius(s)[0])
    return feat
print("invsym builder ready")

## Cell 5: Build ALL features for every structure -> one table

In [ ]:
recs = []
for n, (uid, s) in enumerate(structures.items()):
    nd6 = neighbor_data(s, RC); feat = {"uid": uid}
    # GRDF + Z-weighted GRDF
    for mu in GAUSS_CENTERS:
        feat[f"grdf_gauss_{mu:.1f}_mean"], feat[f"grdf_gauss_{mu:.1f}_std"] = grdf(nd6, lambda r, mu=mu: b_gauss(r, mu))
        feat[f"zw_grdf_gauss_{mu:.1f}_mean"], feat[f"zw_grdf_gauss_{mu:.1f}_std"] = grdf(nd6, lambda r, mu=mu: b_gauss(r, mu), weighted=True)
    for i, k in enumerate(COS_K, 1):
        feat[f"grdf_cos_k{i}_mean"], feat[f"grdf_cos_k{i}_std"] = grdf(nd6, lambda r, k=k: b_cos(r, k))
    for nb_ in BESSEL_N:
        feat[f"grdf_bessel_n{nb_}_mean"], feat[f"grdf_bessel_n{nb_}_std"] = grdf(nd6, lambda r, n=nb_: b_bessel(r, n))
    a = afs(nd6)
    for m in AFS_HARMONICS: feat[f"afs_m{m}_mean"], feat[f"afs_m{m}_std"] = a[m]
    feat.update(simple_geom(nd6)); feat.update(elemental_props(s))
    feat.update(build_invsym(s))
    recs.append(feat)
    if (n+1) % 40 == 0: print(f"  {n+1}/{len(structures)}")
df_feat = pd.DataFrame(recs)
work = master.merge(df_feat, on="uid")

INVSYM = [c for c in df_feat.columns if c.startswith(("inv_","mir_","glob_","buckling","topbot","wrap"))]
GRDF   = [c for c in df_feat.columns if c.startswith(("grdf_","zw_grdf_","afs_","coord_","bond_","angle_"))]
ELEM   = ["max_Z","max_Z4","max_mass","radius_mean","radius_diff","n_elements","mean_Z","sum_Z4"]
BASELINE = ["E_pfrac_VBM","E_pfrac_CBM"]
BUCKLING_99_ONLY = ["buckling"]
print(f"\nwork: {len(work)} rows | invsym={len(INVSYM)} grdf={len(GRDF)} elem={len(ELEM)}")
work.to_csv(os.path.join(RESULTS_DIR, "champion_features.csv"), index=False)

## Cell 6: Modeling helpers - 5-fold for the search, LOO to confirm

Same XGBRegressor config as the project. `r2_5f` (fast, used in the exhaustive search) and `r2_loo`
(used to confirm finalists). `slice_frame` selects 99 / 124 / 200.

In [ ]:
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

XGB = dict(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8,
           colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbosity=0)
_kf = KFold(n_splits=5, shuffle=True, random_state=42)

def r2_5f(fr, feats, y):
    X = fr[feats].fillna(0).values
    return r2_score(y, cross_val_predict(XGBRegressor(**XGB), X, y, cv=_kf))

def r2_loo(fr, feats, y):
    X = fr[feats].fillna(0).values
    yl = cross_val_predict(XGBRegressor(**XGB), X, y, cv=LeaveOneOut())
    return r2_score(y, yl), mean_absolute_error(y, yl)

def slice_frame(name):
    if name == "99":  return work[work["set"] == "rashba"].reset_index(drop=True)
    if name == "124": return work[work["is_split"] == 1].reset_index(drop=True)
    return work.reset_index(drop=True)
print("helpers ready")

## Cell 7: Build the champion pool (capped at 15)

best ~10 invsym by |alpha-corr on the 99| + the 2 baseline + a few top GRDF/elemental by the same
ranking, deduped and capped at 15 so C(15,6)=5005 is tractable. Pool is built per target so the
alpha_max and alpha_mean runs each get features ranked for *their* target.

In [ ]:
POOL_CAP = 15

def rank_by_corr(fr, feats, y):
    out = []
    for f in feats:
        x = fr[f].values
        rho = abs(spearmanr(x, y).correlation) if np.std(x) > 0 else 0.0
        out.append((rho if rho == rho else 0.0, f))   # nan-safe
    return [f for _, f in sorted(out, reverse=True)]

def build_pool(slice_name, target_col):
    fr = slice_frame(slice_name); y = fr[target_col].values
    invsym = [f for f in INVSYM if not (slice_name != "99" and f in BUCKLING_99_ONLY)]
    top_inv  = rank_by_corr(fr, invsym, y)[:10]
    top_grdf = rank_by_corr(fr, GRDF, y)[:2]
    top_elem = rank_by_corr(fr, ELEM, y)[:2]
    pool = list(dict.fromkeys(BASELINE + top_inv + top_grdf + top_elem))[:POOL_CAP]
    return fr, y, pool, top_inv

# preview the pool for each run
for sl, tcol in [("99", "alpha"), ("99", "alpha_mean"), ("124", "alpha")]:
    _, _, pool, top_inv = build_pool(sl, tcol)
    print(f"\nslice {sl} target {tcol}: pool ({len(pool)}):")
    print("  ", pool)

## Cell 8: Exhaustive 4-6 + greedy 7-10 (the heavy run)

For each (slice, target): exhaustively evaluate every size-4, size-5, size-6 subset of the pool with
5-fold R2 and keep the best of each size; then continue greedily from the best-6 up to 10. Finally
confirm the single best combination found with LOO. This is the combinatorial search you asked for -
no greedy until 6, greedy after.

In [ ]:
def exhaustive_then_greedy(fr, y, pool, sizes_exhaustive=(4, 5, 6), greedy_upto=10):
    results = []
    best_overall = (-np.inf, None)
    best6 = None
    for k in sizes_exhaustive:
        bk = (-np.inf, None)
        for combo in itertools.combinations(pool, k):
            r2 = r2_5f(fr, list(combo), y)
            if r2 > bk[0]:
                bk = (r2, list(combo))
        results.append((k, bk[0], bk[1]))
        if bk[0] > best_overall[0]:
            best_overall = bk
        if k == 6:
            best6 = bk[1]
    # greedy from best-6
    chosen = list(best6) if best6 else (list(best_overall[1]) if best_overall[1] else [])
    while len(chosen) < greedy_upto and len(chosen) < len(pool):
        rem = [f for f in pool if f not in chosen]
        cand = sorted(((r2_5f(fr, chosen + [f], y), f) for f in rem), reverse=True)
        chosen.append(cand[0][1])
        r2 = r2_5f(fr, chosen, y)
        results.append((len(chosen), r2, list(chosen)))
        if r2 > best_overall[0]:
            best_overall = (r2, list(chosen))
    return results, best_overall

all_runs = {}
for sl, tcol in [("99", "alpha"), ("99", "alpha_mean"), ("124", "alpha")]:
    fr, y, pool, _ = build_pool(sl, tcol)
    print(f"\n========== SLICE {sl}  target {tcol}  (n={len(fr)}, pool={len(pool)}) ==========")
    results, best = exhaustive_then_greedy(fr, y, pool)
    for k, r2, combo in results:
        tag = "exhaustive" if k <= 6 else "greedy"
        print(f"  size {k:2d} [{tag:10s}] best 5f R2 = {r2:.3f}")
    # confirm the overall best with LOO
    loo_r2, loo_mae = r2_loo(fr, best[1], y)
    print(f"  -> overall best: 5f R2 = {best[0]:.3f} | LOO R2 = {loo_r2:.3f} | MAE = {loo_mae:.3f}")
    print(f"     features ({len(best[1])}): {best[1]}")
    all_runs[f"{sl}_{tcol}"] = dict(best_5f=best[0], loo_r2=loo_r2, loo_mae=loo_mae,
                                    n_feat=len(best[1]), features=best[1])

## Cell 9: Summary table

In [ ]:
print("================ CHAMPION-POOL SUMMARY ================")
print(f"  {'run':14s} {'5f R2':>7} {'LOO R2':>7} {'MAE':>6} {'nfeat':>6}")
for key, d in all_runs.items():
    print(f"  {key:14s} {d['best_5f']:7.3f} {d['loo_r2']:7.3f} {d['loo_mae']:6.3f} {d['n_feat']:6d}")
pd.DataFrame([{"run": k, **{kk: vv for kk, vv in d.items() if kk != "features"}}
              for k, d in all_runs.items()]).to_csv(
    os.path.join(RESULTS_DIR, "champion_summary.csv"), index=False)
import json as _json
with open(os.path.join(RESULTS_DIR, "champion_features_chosen.json"), "w") as fh:
    _json.dump({k: d["features"] for k, d in all_runs.items()}, fh, indent=2)
print("\nsaved champion_summary.csv + champion_features_chosen.json")

## What to read

- The per-size lines show whether an exhaustive 4/5/6 combo beats what greedy alone would have found,
  and where adding features stops helping.
- The summary compares the best pooled model (invsym + baseline + GRDF + elemental) on **99 alpha_max**,
  **99 alpha_mean**, and **124 alpha_max** - all LOO-confirmed - against the ~0.66 environment-only ceiling.
- The chosen feature lists (JSON) tell us which descriptor groups actually make the final model: if
  invsym features survive into the champion combos, broken-inversion-symmetry earned its place in the report.

Note: exhaustive search uses 5-fold (faster, slightly optimistic); the headline number is always the
LOO confirmation, consistent with the rest of the project.